# Fine-tune ECAPA-TDNN on compact VoxVietnam

Protocol:

- Train AAM-Softmax on the closed-set classifier-train partition.
- Compare "frozen ECAPA-TDNN" baseline after checkpoint selection.
- Validation and test speakers are unseen and pairwise disjoint.
- Select checkpoint and cosine threshold only from validation EER/minDCF.
- After restoring one best checkpoint, cache every audio embedding once.
- Evaluate closed-set LinearSVC, claimed-centroid verification, and
  gallery-centroid open-set identification with validation-only selection.

Attach the private Kaggle dataset generated by
`prepare-voxvietnam-on-kaggle.ipynb`. Do not publish derived audio unless
VoxVietnam gated-access terms permit redistribution.


## 1. Install and imports


In [ ]:
!pip install -q speechbrain soundfile psutil seaborn


In [ ]:
import gc
import json
import os
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from sklearn.metrics import f1_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.lobes.models.ECAPA_TDNN import Classifier
from speechbrain.nnet.losses import AdditiveAngularMargin, LogSoftmaxWrapper

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


## 2. Configuration — edit only this cell

All dataset, audio, hardware, batching, fine-tuning, and artifact
settings live here. Later cells consume these values without redeclaring
them. Safe default targets one T4 even when Kaggle exposes T4 x2.


In [ ]:
# Reproducibility and dataset discovery
SEED = 42
EXPECTED_DATASET = "voxvietnam_ecapa_three_task_v1"
LABEL_COL = "normalized_speaker_id"

# Audio and DataLoader
TARGET_SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 3
SEGMENT_SAMPLES = SEGMENT_SECONDS * TARGET_SAMPLE_RATE
NUM_WORKERS = 0

# Hardware and memory-safe batching for one 15 GiB T4
AVAILABLE_GPU_COUNT = torch.cuda.device_count()
GPU_COUNT = min(1, AVAILABLE_GPU_COUNT)
DEVICE = torch.device("cuda:0" if GPU_COUNT else "cpu")
USE_AMP = DEVICE.type == "cuda"
PER_GPU_BATCH = 2
BATCH_SIZE = PER_GPU_BATCH * max(GPU_COUNT, 1)
TARGET_EFFECTIVE_BATCH = 32
GRAD_ACCUM = TARGET_EFFECTIVE_BATCH // BATCH_SIZE
assert TARGET_EFFECTIVE_BATCH % BATCH_SIZE == 0

# Fine-tuning
ENCODER_LEARNING_RATE = 1e-5
CLASSIFIER_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 2e-6
MAX_EPOCHS = 30
WARMUP_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 3

# Outputs
WORKING_DIR = Path("/kaggle/working")
CHECKPOINT_EVAL_INTERVAL = 2
MAX_ROLLING_CHECKPOINTS = 5
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
BEST_CLOSED_PATH = CHECKPOINT_DIR / "best_closed.pt"
BEST_VERIFICATION_PATH = CHECKPOINT_DIR / "best_verification.pt"
BEST_OPEN_PATH = CHECKPOINT_DIR / "best_open.pt"
BEST_BALANCED_PATH = CHECKPOINT_DIR / "best_balanced.pt"
LATEST_PATH = CHECKPOINT_DIR / "latest.pt"
MAX_CLOSED_F1_DROP = 0.005
TRAINING_CONFIG_PATH = WORKING_DIR / "ecapa_training_config.json"
TRAINING_HISTORY_PATH = WORKING_DIR / "ecapa_training_history.csv"
FROZEN_EVALUATION_DIR = WORKING_DIR / "evaluation/frozen"
FINETUNED_EVALUATION_DIR = WORKING_DIR / "evaluation/finetuned"
SUMMARY_PATH = WORKING_DIR / "three_task_summary.csv"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "device": str(DEVICE),
    "visible_gpu_count": AVAILABLE_GPU_COUNT,
    "used_gpu_count": GPU_COUNT,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
})


## 3. Locate and validate VoxVietnam dataset


In [ ]:
def find_dataset_root():
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for manifest_path in search_root.rglob("manifest.json"):
            try:
                manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                continue
            if manifest.get("dataset") == EXPECTED_DATASET:
                matches.append((manifest_path.parent, manifest))
    if not matches:
        raise FileNotFoundError(
            "Attach VoxVietnam Kaggle output containing manifest.json with "
            f"dataset={EXPECTED_DATASET!r}."
        )
    if len(matches) > 1:
        print("Multiple matching datasets found; using:", matches[0][0])
    return matches[0]


DATASET_ROOT, DATASET_MANIFEST = find_dataset_root()
train_metadata = pd.read_csv(DATASET_ROOT / "train/metadata.csv")
valid_df = pd.read_csv(DATASET_ROOT / "validation/metadata.csv")
test_df = pd.read_csv(DATASET_ROOT / "test/metadata.csv")
closed_train_protocol = pd.read_csv(DATASET_ROOT / "protocols/closed_set/classifier_train.csv")
closed_valid_protocol = pd.read_csv(DATASET_ROOT / "protocols/closed_set/validation_queries.csv")
closed_test_protocol = pd.read_csv(DATASET_ROOT / "protocols/closed_set/test_queries.csv")
valid_enrollment_protocol = pd.read_csv(DATASET_ROOT / "protocols/verification/validation_enrollment.csv")
valid_trials = pd.read_csv(DATASET_ROOT / "protocols/verification/validation_trials.csv")
test_enrollment_protocol = pd.read_csv(DATASET_ROOT / "protocols/verification/test_enrollment.csv")
test_trials = pd.read_csv(DATASET_ROOT / "protocols/verification/test_trials.csv")
open_valid_gallery = pd.read_csv(DATASET_ROOT / "protocols/open_set/validation_gallery.csv")
open_valid_queries = pd.read_csv(DATASET_ROOT / "protocols/open_set/validation_queries.csv")
open_test_gallery = pd.read_csv(DATASET_ROOT / "protocols/open_set/test_gallery.csv")
open_test_queries = pd.read_csv(DATASET_ROOT / "protocols/open_set/test_queries.csv")

all_metadata = pd.concat([train_metadata, valid_df, test_df], ignore_index=True)
metadata_lookup = all_metadata.set_index("audio_path", drop=False)


def attach_metadata(protocol, path_column="audio_path"):
    paths = protocol[path_column].astype(str)
    missing = sorted(set(paths) - set(metadata_lookup.index))
    assert not missing, f"Protocol paths absent from metadata: {missing[:3]}"
    return metadata_lookup.loc[paths].reset_index(drop=True)


train_df = attach_metadata(closed_train_protocol)
closed_valid_df = attach_metadata(closed_valid_protocol)
closed_test_df = attach_metadata(closed_test_protocol)
evaluation_df = all_metadata.drop_duplicates("audio_path").reset_index(drop=True)
THREE_TASK_PROTOCOLS = {
    "closed_train": closed_train_protocol,
    "closed_validation": closed_valid_protocol,
    "closed_test": closed_test_protocol,
    "verification_validation_enrollment": valid_enrollment_protocol,
    "verification_validation_trials": valid_trials,
    "verification_test_enrollment": test_enrollment_protocol,
    "verification_test_trials": test_trials,
    "open_validation_gallery": open_valid_gallery,
    "open_validation_queries": open_valid_queries,
    "open_test_gallery": open_test_gallery,
    "open_test_queries": open_test_queries,
}

print("Dataset root:", DATASET_ROOT)
print(json.dumps(DATASET_MANIFEST["splits"], indent=2))


In [ ]:
REQUIRED_METADATA = {
    "audio_id", "audio_path", "speaker_id", LABEL_COL,
    "role", "sample_rate", "checksum",
}
REQUIRED_TRIALS = {
    "trial_id", "enrollment_speaker_id", "query_audio_path",
    "query_speaker_id", "label",
}

for split_name, frame in {
    "train": train_metadata,
    "validation": valid_df,
    "test": test_df,
}.items():
    missing = REQUIRED_METADATA - set(frame.columns)
    assert not missing, f"{split_name} missing columns: {sorted(missing)}"
    assert not frame.empty
    assert frame["audio_path"].is_unique
    assert set(frame["sample_rate"].astype(int)) == {16_000}
    missing_files = [
        value for value in frame["audio_path"]
        if not (DATASET_ROOT / value).is_file()
    ]
    assert not missing_files, f"{split_name} missing audio: {missing_files[:3]}"

for split_name, trials in {
    "validation": valid_trials,
    "test": test_trials,
}.items():
    missing = REQUIRED_TRIALS - set(trials.columns)
    assert not missing, f"{split_name} trials missing: {sorted(missing)}"
    assert set(trials["label"].astype(int)) == {0, 1}

speaker_sets = {
    "train": set(train_metadata[LABEL_COL].astype(str)),
    "validation": set(valid_df[LABEL_COL].astype(str)),
    "test": set(test_df[LABEL_COL].astype(str)),
}
assert speaker_sets["train"].isdisjoint(speaker_sets["validation"])
assert speaker_sets["train"].isdisjoint(speaker_sets["test"])
assert speaker_sets["validation"].isdisjoint(speaker_sets["test"])

assert set(train_df["role"]) == {"TRAIN"}
assert set(valid_df["role"]) == {"ENROLLMENT", "QUERY"}
assert set(test_df["role"]) == {"ENROLLMENT", "QUERY"}
assert set(closed_train_protocol["speaker_id"]) == set(closed_valid_protocol["speaker_id"])
assert set(closed_train_protocol["speaker_id"]) == set(closed_test_protocol["speaker_id"])
closed_path_sets = [
    set(frame["audio_path"])
    for frame in (closed_train_protocol, closed_valid_protocol, closed_test_protocol)
]
assert all(
    left.isdisjoint(right)
    for index, left in enumerate(closed_path_sets)
    for right in closed_path_sets[index + 1:]
)
for gallery, queries in (
    (open_valid_gallery, open_valid_queries),
    (open_test_gallery, open_test_queries),
):
    gallery_speakers = set(gallery["speaker_id"].astype(str))
    assert all(
        (speaker in gallery_speakers) == bool(is_known)
        for speaker, is_known in zip(
            queries["query_speaker_id"].astype(str),
            queries["is_known"].astype(int),
        )
    )

display(pd.DataFrame([
    {
        "split": name,
        "audio": len(frame),
        "speakers": frame[LABEL_COL].nunique(),
    }
    for name, frame in {
        "train": train_metadata,
        "validation": valid_df,
        "test": test_df,
    }.items()
]))
print("Dataset protocol checks passed")


## 4. Audio pipeline


In [ ]:
def load_segment(path, training):
    info = sf.info(path)
    source_frames = round(
        SEGMENT_SAMPLES * info.samplerate / TARGET_SAMPLE_RATE
    )
    max_start = max(0, info.frames - source_frames)
    start = (
        torch.randint(0, max_start + 1, (1,)).item()
        if training and max_start
        else max_start // 2
    )
    audio, sample_rate = sf.read(
        path,
        start=start,
        frames=source_frames,
        dtype="float32",
        always_2d=True,
    )
    waveform = torch.from_numpy(audio.mean(axis=1)).float()
    if sample_rate != TARGET_SAMPLE_RATE:
        waveform = torchaudio.functional.resample(
            waveform, sample_rate, TARGET_SAMPLE_RATE
        )
    if waveform.numel() == 0 or not torch.isfinite(waveform).all():
        raise ValueError(f"Invalid audio: {path}")
    valid_samples = min(waveform.numel(), SEGMENT_SAMPLES)
    waveform = waveform[:SEGMENT_SAMPLES]
    if waveform.numel() < SEGMENT_SAMPLES:
        waveform = F.pad(
            waveform, (0, SEGMENT_SAMPLES - waveform.numel())
        )
    return waveform, valid_samples


class VoxVietnamAudioDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        relative_path = str(row["audio_path"])
        waveform, valid_samples = load_segment(
            DATASET_ROOT / relative_path, self.training
        )
        return (
            waveform,
            valid_samples,
            relative_path,
            str(row[LABEL_COL]),
        )


def collate_audio(batch):
    waveforms, valid_samples, paths, speakers = zip(*batch)
    waveforms = torch.stack(waveforms)
    lengths = torch.tensor(valid_samples, dtype=torch.float32)
    lengths = lengths / waveforms.shape[1]
    return waveforms, lengths, list(paths), list(speakers)


## 5. Checkpoint-selection verification metrics


In [ ]:
def verification_curve(labels, scores, p_target=0.01):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    assert labels.shape == scores.shape
    assert set(np.unique(labels)) == {0, 1}
    assert np.isfinite(scores).all()

    fpr, tpr, thresholds = roc_curve(
        labels, scores, pos_label=1, drop_intermediate=False
    )
    fnr = 1.0 - tpr
    eer_index = int(np.argmin(np.abs(fpr - fnr)))
    eer = float((fpr[eer_index] + fnr[eer_index]) / 2.0)

    dcf = p_target * fnr + (1.0 - p_target) * fpr
    normalizer = min(p_target, 1.0 - p_target)
    min_dcf_index = int(np.argmin(dcf))

    far_mask = fpr <= 0.01
    tar_at_far_1pct = float(tpr[far_mask].max()) if far_mask.any() else 0.0
    return {
        "eer": eer,
        "eer_threshold": float(thresholds[eer_index]),
        "min_dcf": float(dcf[min_dcf_index] / normalizer),
        "min_dcf_threshold": float(thresholds[min_dcf_index]),
        "tar_at_far_1pct": tar_at_far_1pct,
    }


def rates_at_threshold(labels, scores, threshold):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = np.asarray(scores) >= threshold
    positives = labels == 1
    negatives = ~positives
    far = float(predictions[negatives].mean())
    frr = float((~predictions[positives]).mean())
    return {
        "threshold": float(threshold),
        "far": far,
        "frr": frr,
        "tar": float(1.0 - frr),
    }


def score_trials(frame, embeddings, trials):
    assert len(frame) == len(embeddings)
    path_to_embedding = {
        str(path): embedding
        for path, embedding in zip(frame["audio_path"], embeddings)
    }
    enrollment = frame[frame["role"] == "ENROLLMENT"]
    centroids = {}
    for speaker, group in enrollment.groupby(LABEL_COL):
        vectors = np.stack([
            path_to_embedding[str(path)] for path in group["audio_path"]
        ])
        centroid = vectors.mean(axis=0)
        centroids[str(speaker)] = centroid / np.linalg.norm(centroid)

    scores = []
    for row in trials.itertuples(index=False):
        centroid = centroids[str(row.enrollment_speaker_id)]
        query = path_to_embedding[str(row.query_audio_path)]
        scores.append(float(np.dot(centroid, query)))
    return np.asarray(scores, dtype=np.float64)


## 6. Standalone three-task evaluator

This generated cell embeds the tested `src/speaker/evaluation.py` snapshot. Kaggle needs only this notebook and the processed dataset.


In [ ]:
"""Reusable evaluation for ECAPA speaker identification and verification.

The functions in this module operate on already-extracted embeddings.  This
keeps protocol decisions testable and lets notebooks cache each audio embedding
once before applying the three task-specific decision layers.
"""

import json
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.svm import LinearSVC


UNKNOWN_SPEAKER = "UNKNOWN"


def _matrix(values: Any, *, name: str) -> np.ndarray:
    matrix = np.asarray(values, dtype=np.float64)
    if matrix.ndim == 1:
        matrix = matrix.reshape(1, -1)
    if matrix.ndim != 2 or not matrix.size:
        raise ValueError(f"{name} must be a non-empty 1-D or 2-D array")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} must contain only finite values")
    return matrix


def _binary(labels: Any, *, name: str = "labels") -> np.ndarray:
    values = np.asarray(labels, dtype=np.int64).reshape(-1)
    if set(np.unique(values)) != {0, 1}:
        raise ValueError(f"{name} must contain both classes 0 and 1")
    return values


def _scores(values: Any, *, expected: int | None = None) -> np.ndarray:
    scores = np.asarray(values, dtype=np.float64).reshape(-1)
    if not scores.size or not np.isfinite(scores).all():
        raise ValueError("scores must be non-empty and finite")
    if expected is not None and scores.size != expected:
        raise ValueError("labels and scores must have equal length")
    return scores


def l2_normalize(values: Any) -> np.ndarray:
    """Return row-wise L2-normalized embeddings; reject zero/non-finite rows."""

    matrix = _matrix(values, name="embeddings")
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(norms <= np.finfo(np.float64).eps):
        raise ValueError("embeddings contain a zero-norm row")
    normalized = matrix / norms
    return normalized[0] if np.asarray(values).ndim == 1 else normalized


def build_centroids(embeddings: Any, speaker_ids: Sequence[Any]) -> dict[str, np.ndarray]:
    """Build normalized mean centroids for every enrolled speaker."""

    vectors = l2_normalize(embeddings)
    if vectors.ndim == 1:
        vectors = vectors.reshape(1, -1)
    speakers = np.asarray([str(value) for value in speaker_ids], dtype=object)
    if len(speakers) != len(vectors) or not len(speakers):
        raise ValueError("speaker_ids must match the number of embeddings")
    if any(not value for value in speakers):
        raise ValueError("speaker_ids must be non-empty")
    centroids = {}
    for speaker in sorted(set(speakers)):
        centroids[speaker] = l2_normalize(vectors[speakers == speaker].mean(axis=0))
    return centroids


def score_cosine_trials(
    embeddings_by_path: Mapping[str, Any],
    centroids: Mapping[str, Any],
    trials: Iterable[Mapping[str, Any]],
) -> np.ndarray:
    """Score claimed-centroid verification trials in input order."""

    normalized_centroids = {
        str(speaker): l2_normalize(vector)
        for speaker, vector in centroids.items()
    }
    output = []
    for trial in trials:
        speaker = str(trial.get("enrollment_speaker_id", ""))
        path = str(trial.get("query_audio_path", ""))
        if speaker not in normalized_centroids:
            raise ValueError(f"Missing enrollment centroid: {speaker}")
        if path not in embeddings_by_path:
            raise ValueError(f"Missing query embedding: {path}")
        query = l2_normalize(embeddings_by_path[path])
        if query.shape != normalized_centroids[speaker].shape:
            raise ValueError("Query and centroid dimensions do not match")
        output.append(float(query @ normalized_centroids[speaker]))
    if not output:
        raise ValueError("trials must not be empty")
    return np.asarray(output, dtype=np.float64)


def rates_at_threshold(labels: Any, scores: Any, threshold: float) -> dict[str, float]:
    """Compute FAR/FRR/TAR; equality is accepted (score >= threshold)."""

    binary = _binary(labels)
    values = _scores(scores, expected=len(binary))
    if not np.isfinite(threshold):
        raise ValueError("threshold must be finite")
    accepted = values >= float(threshold)
    positives = binary == 1
    negatives = ~positives
    far = float(accepted[negatives].mean())
    frr = float((~accepted[positives]).mean())
    return {"threshold": float(threshold), "far": far, "frr": frr, "tar": 1.0 - frr}


def verification_metrics(labels: Any, scores: Any, p_target: float = 0.01) -> dict[str, float]:
    """Return threshold-independent EER/minDCF and validation thresholds."""

    binary = _binary(labels)
    values = _scores(scores, expected=len(binary))
    if not 0.0 < p_target < 1.0:
        raise ValueError("p_target must be between zero and one")
    fpr, tpr, thresholds = roc_curve(binary, values, pos_label=1, drop_intermediate=False)
    fnr = 1.0 - tpr
    eer_index = int(np.argmin(np.abs(fpr - fnr)))
    dcf = p_target * fnr + (1.0 - p_target) * fpr
    min_dcf_index = int(np.argmin(dcf))
    far_mask = fpr <= 0.01
    return {
        "eer": float((fpr[eer_index] + fnr[eer_index]) / 2.0),
        "eer_threshold": float(thresholds[eer_index]),
        "min_dcf": float(dcf[min_dcf_index] / min(p_target, 1.0 - p_target)),
        "min_dcf_threshold": float(thresholds[min_dcf_index]),
        "tar_at_far_1pct": float(tpr[far_mask].max()) if far_mask.any() else 0.0,
    }


def select_closed_set_svm(
    train_embeddings: Any,
    train_labels: Sequence[Any],
    validation_embeddings: Any,
    validation_labels: Sequence[Any],
    *,
    c_values: Sequence[float] = (0.01, 0.1, 1.0, 10.0),
    class_weights: Sequence[str | None] = (None, "balanced"),
    random_state: int = 42,
) -> dict[str, Any]:
    """Select LinearSVC hyperparameters using validation labels only."""

    train_x = _matrix(train_embeddings, name="train_embeddings")
    valid_x = _matrix(validation_embeddings, name="validation_embeddings")
    train_y = np.asarray([str(value) for value in train_labels], dtype=object)
    valid_y = np.asarray([str(value) for value in validation_labels], dtype=object)
    if len(train_x) != len(train_y) or len(valid_x) != len(valid_y):
        raise ValueError("embedding and label lengths must match")
    if train_x.shape[1] != valid_x.shape[1]:
        raise ValueError("train and validation embedding dimensions must match")
    if len(set(train_y)) < 2 or not set(valid_y).issubset(set(train_y)):
        raise ValueError("closed-set labels require at least two train classes and no unseen validation class")
    candidates = []
    for c_value in c_values:
        if not np.isfinite(c_value) or c_value <= 0:
            raise ValueError("c_values must be positive and finite")
        for class_weight in class_weights:
            model = LinearSVC(
                C=float(c_value), class_weight=class_weight,
                random_state=random_state, dual="auto", max_iter=20_000,
            ).fit(train_x, train_y)
            predictions = model.predict(valid_x)
            macro_f1 = float(f1_score(valid_y, predictions, average="macro", zero_division=0))
            accuracy = float(accuracy_score(valid_y, predictions))
            candidates.append((macro_f1, accuracy, -float(c_value), class_weight is None, model, c_value, class_weight))
    best = max(candidates, key=lambda row: row[:4])
    return {
        "model": best[4],
        "selected_c": float(best[5]),
        "class_weight": best[6],
        "validation": {"macro_f1": best[0], "accuracy": best[1]},
    }


def evaluate_closed_set(model: Any, embeddings: Any, labels: Sequence[Any]) -> dict[str, Any]:
    """Evaluate a previously selected closed-set classifier."""

    features = _matrix(embeddings, name="test_embeddings")
    truth = np.asarray([str(value) for value in labels], dtype=object)
    if len(features) != len(truth):
        raise ValueError("test embedding and label lengths must match")
    predictions = np.asarray(model.predict(features), dtype=object)
    classes = np.asarray([str(value) for value in model.classes_], dtype=object)
    if not set(truth).issubset(set(classes)):
        raise ValueError("closed-set test labels contain an unseen class")
    recalls = recall_score(truth, predictions, labels=classes, average=None, zero_division=0)
    return {
        "predictions": predictions,
        "classes": classes,
        "confusion_matrix": confusion_matrix(truth, predictions, labels=classes),
        "per_speaker_recall": {speaker: float(value) for speaker, value in zip(classes, recalls)},
        "metrics": {
            "accuracy": float(accuracy_score(truth, predictions)),
            "macro_f1": float(f1_score(truth, predictions, average="macro", zero_division=0)),
        },
    }


def evaluate_closed_validation(train_embeddings: Any, train_labels: Sequence[Any], validation_embeddings: Any, validation_labels: Sequence[Any]) -> dict[str, Any]:
    """Fit classifier on closed-train and score closed-validation only."""
    selection = select_closed_set_svm(train_embeddings, train_labels, validation_embeddings, validation_labels)
    return {"metrics": selection["validation"], "model": selection["model"], "selected_c": selection["selected_c"], "class_weight": selection["class_weight"]}


def evaluate_verification_validation(embedding_cache: Mapping[str, Any], enrollment: Any, trials: Any) -> dict[str, Any]:
    centroids = build_centroids(_protocol_vectors(embedding_cache, enrollment), enrollment["speaker_id"])
    scores = score_cosine_trials(embedding_cache, centroids, trials.to_dict("records"))
    metrics = verification_metrics(trials["label"], scores)
    return {"metrics": metrics, "scores": scores}


def evaluate_open_validation(embedding_cache: Mapping[str, Any], gallery: Any, queries: Any) -> dict[str, Any]:
    centroids = build_centroids(_protocol_vectors(embedding_cache, gallery), gallery["speaker_id"])
    candidates, scores = predict_open_set(_protocol_vectors(embedding_cache, queries, "query_audio_path"), centroids, threshold=-1.0)
    threshold = select_open_set_threshold(queries["is_known"], scores)
    return {"metrics": open_set_metrics(queries["query_speaker_id"], queries["is_known"], candidates, scores, threshold=threshold), "scores": scores, "threshold": threshold}


def predict_open_set(
    query_embeddings: Any,
    centroids: Mapping[str, Any],
    threshold: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Return maximum-centroid identities, rejecting scores below threshold."""

    queries = l2_normalize(query_embeddings)
    if queries.ndim == 1:
        queries = queries.reshape(1, -1)
    if not centroids:
        raise ValueError("gallery centroids must not be empty")
    speakers = np.asarray(sorted(str(value) for value in centroids), dtype=object)
    gallery = np.stack([l2_normalize(centroids[speaker]) for speaker in speakers])
    if queries.shape[1] != gallery.shape[1] or not np.isfinite(threshold):
        raise ValueError("query/gallery dimensions and threshold must be valid")
    similarity = queries @ gallery.T
    winners = similarity.argmax(axis=1)
    scores = similarity[np.arange(len(queries)), winners]
    identities = speakers[winners].copy()
    identities[scores < float(threshold)] = UNKNOWN_SPEAKER
    return identities, scores.astype(np.float64)


def select_open_set_threshold(is_known: Any, max_scores: Any) -> float:
    """Select the equal-error known/unknown rejection threshold on validation."""

    labels = _binary(is_known, name="is_known")
    scores = _scores(max_scores, expected=len(labels))
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1, drop_intermediate=False)
    index = int(np.argmin(np.abs(fpr - (1.0 - tpr))))
    return float(thresholds[index])


def open_set_metrics(
    true_speaker_ids: Sequence[Any],
    is_known: Any,
    predicted_speaker_ids: Sequence[Any],
    max_scores: Any,
    *,
    threshold: float,
    far_target: float = 0.01,
) -> dict[str, float]:
    """Compute known ID, unknown rejection, AUROC, FAR/FRR, and DIR@FAR."""

    truth = np.asarray([str(value) for value in true_speaker_ids], dtype=object)
    predictions = np.asarray([str(value) for value in predicted_speaker_ids], dtype=object)
    known = _binary(is_known, name="is_known")
    scores = _scores(max_scores, expected=len(known))
    if len(truth) != len(known) or len(predictions) != len(known):
        raise ValueError("open-set arrays must have equal length")
    if not 0.0 <= far_target <= 1.0:
        raise ValueError("far_target must be between zero and one")
    known_mask = known == 1
    unknown_mask = ~known_mask
    accepted = scores >= float(threshold)
    candidate_correct = predictions[known_mask] == truth[known_mask]
    far = float(accepted[unknown_mask].mean())
    frr = float((~accepted[known_mask]).mean())
    unknown_scores = np.sort(scores[unknown_mask])[::-1]
    allowed_false_accepts = int(np.floor(far_target * len(unknown_scores)))
    fixed_threshold = (
        float("inf") if allowed_false_accepts == 0
        else float(np.nextafter(unknown_scores[allowed_false_accepts - 1], np.inf))
    )
    # With zero allowed false accepts, the highest unknown score can still be
    # rejected while known scores strictly above it contribute to DIR.
    if allowed_false_accepts == 0:
        fixed_threshold = float(np.nextafter(unknown_scores[0], np.inf))
    dir_value = float(((scores[known_mask] >= fixed_threshold) & candidate_correct).mean())
    return {
        "threshold": float(threshold),
        "known_identification_accuracy": float(
            (accepted[known_mask] & candidate_correct).mean()
        ),
        "unknown_rejection_rate": float((~accepted[unknown_mask]).mean()),
        "known_unknown_auroc": float(roc_auc_score(known, scores)),
        "far": far,
        "frr": frr,
        "dir_at_far_1pct": dir_value,
        "dir_far_target": float(far_target),
        "dir_threshold": fixed_threshold,
    }


def _jsonable(value: Any) -> Any:
    if isinstance(value, Mapping):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [_jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    raise TypeError(f"Metric value is not JSON serializable: {type(value).__name__}")


def write_metrics_json(path: str | Path, metrics: Mapping[str, Any]) -> None:
    """Serialize machine-readable metrics with stable formatting."""

    Path(path).write_text(
        json.dumps(_jsonable(metrics), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def _protocol_vectors(
    cache: Mapping[str, Any],
    protocol: Any,
    path_column: str = "audio_path",
) -> np.ndarray:
    paths = protocol[path_column].astype(str)
    missing = sorted(set(paths) - set(cache))
    if missing:
        raise ValueError(f"Missing cached protocol embeddings: {missing[:3]}")
    return np.stack([cache[str(path)] for path in paths])


def evaluate_three_tasks(
    model_name: str,
    embedding_cache: Mapping[str, Any],
    protocols: Mapping[str, Any],
    *,
    output_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Evaluate closed-set ID, verification, and open-set ID for one encoder.

    All model and threshold selection uses validation data. Test data is read
    only after selection. When ``output_dir`` is provided, task metrics and
    prediction artifacts are written under that model-specific directory.
    """

    required = {
        "closed_train",
        "closed_validation",
        "closed_test",
        "verification_validation_enrollment",
        "verification_validation_trials",
        "verification_test_enrollment",
        "verification_test_trials",
        "open_validation_gallery",
        "open_validation_queries",
        "open_test_gallery",
        "open_test_queries",
    }
    missing_protocols = sorted(required - set(protocols))
    if missing_protocols:
        raise ValueError(f"Missing three-task protocols: {missing_protocols}")

    closed_train = protocols["closed_train"]
    closed_validation = protocols["closed_validation"]
    closed_test = protocols["closed_test"]
    verification_validation_enrollment = protocols[
        "verification_validation_enrollment"
    ]
    verification_validation_trials = protocols["verification_validation_trials"]
    verification_test_enrollment = protocols["verification_test_enrollment"]
    verification_test_trials = protocols["verification_test_trials"]
    open_validation_gallery = protocols["open_validation_gallery"]
    open_validation_queries = protocols["open_validation_queries"]
    open_test_gallery = protocols["open_test_gallery"]
    open_test_queries = protocols["open_test_queries"]

    closed_selection = select_closed_set_svm(
        _protocol_vectors(embedding_cache, closed_train),
        closed_train["speaker_id"],
        _protocol_vectors(embedding_cache, closed_validation),
        closed_validation["speaker_id"],
    )
    closed_test_result = evaluate_closed_set(
        closed_selection["model"],
        _protocol_vectors(embedding_cache, closed_test),
        closed_test["speaker_id"],
    )
    closed_metrics = {
        "model": model_name,
        "selected_c": closed_selection["selected_c"],
        "class_weight": closed_selection["class_weight"],
        "validation": closed_selection["validation"],
        "test": closed_test_result["metrics"],
        "test_per_speaker_recall": closed_test_result["per_speaker_recall"],
    }

    validation_centroids = build_centroids(
        _protocol_vectors(embedding_cache, verification_validation_enrollment),
        verification_validation_enrollment["speaker_id"],
    )
    validation_verification_scores = score_cosine_trials(
        embedding_cache,
        validation_centroids,
        verification_validation_trials.to_dict("records"),
    )
    validation_verification = verification_metrics(
        verification_validation_trials["label"],
        validation_verification_scores,
    )
    verification_threshold = validation_verification["min_dcf_threshold"]
    test_centroids = build_centroids(
        _protocol_vectors(embedding_cache, verification_test_enrollment),
        verification_test_enrollment["speaker_id"],
    )
    test_verification_scores = score_cosine_trials(
        embedding_cache,
        test_centroids,
        verification_test_trials.to_dict("records"),
    )
    test_verification = verification_metrics(
        verification_test_trials["label"], test_verification_scores
    )
    test_verification.update(
        rates_at_threshold(
            verification_test_trials["label"],
            test_verification_scores,
            verification_threshold,
        )
    )
    verification_result = {
        "model": model_name,
        "validation": validation_verification,
        "test": test_verification,
    }

    validation_gallery_centroids = build_centroids(
        _protocol_vectors(embedding_cache, open_validation_gallery),
        open_validation_gallery["speaker_id"],
    )
    validation_candidates, validation_scores = predict_open_set(
        _protocol_vectors(
            embedding_cache, open_validation_queries, "query_audio_path"
        ),
        validation_gallery_centroids,
        threshold=-1.0,
    )
    open_threshold = select_open_set_threshold(
        open_validation_queries["is_known"], validation_scores
    )
    validation_open_metrics = open_set_metrics(
        open_validation_queries["query_speaker_id"],
        open_validation_queries["is_known"],
        validation_candidates,
        validation_scores,
        threshold=open_threshold,
    )
    test_gallery_centroids = build_centroids(
        _protocol_vectors(embedding_cache, open_test_gallery),
        open_test_gallery["speaker_id"],
    )
    test_candidates, test_scores = predict_open_set(
        _protocol_vectors(embedding_cache, open_test_queries, "query_audio_path"),
        test_gallery_centroids,
        threshold=-1.0,
    )
    test_open_metrics = open_set_metrics(
        open_test_queries["query_speaker_id"],
        open_test_queries["is_known"],
        test_candidates,
        test_scores,
        threshold=open_threshold,
    )
    open_result = {
        "model": model_name,
        "validation": validation_open_metrics,
        "test": test_open_metrics,
    }

    if output_dir is not None:
        import pandas as pd

        destination = Path(output_dir)
        destination.mkdir(parents=True, exist_ok=True)
        write_metrics_json(destination / "verification_metrics.json", verification_result)
        verification_output = verification_test_trials.copy()
        verification_output["score"] = test_verification_scores
        verification_output["accepted"] = (
            test_verification_scores >= verification_threshold
        )
        verification_output.to_csv(
            destination / "verification_trial_scores.csv", index=False
        )

        write_metrics_json(destination / "closed_set_metrics.json", closed_metrics)
        closed_output = closed_test.copy()
        closed_output["predicted_speaker_id"] = closed_test_result["predictions"]
        closed_output.to_csv(destination / "closed_set_predictions.csv", index=False)
        pd.DataFrame(
            closed_test_result["confusion_matrix"],
            index=closed_test_result["classes"],
            columns=closed_test_result["classes"],
        ).to_csv(destination / "closed_set_confusion_matrix.csv")

        write_metrics_json(destination / "open_set_metrics.json", open_result)
        open_output = open_test_queries.copy()
        open_output["predicted_speaker_id"] = np.where(
            test_scores >= open_threshold, test_candidates, UNKNOWN_SPEAKER
        )
        open_output["max_score"] = test_scores
        open_output["threshold"] = open_threshold
        open_output.to_csv(destination / "open_set_predictions.csv", index=False)

    return {
        "model": model_name,
        "closed_set": closed_metrics,
        "verification": verification_result,
        "open_set": open_result,
    }


def summarize_three_task_results(results: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Build comparable summary rows for frozen and fine-tuned encoders."""

    return [
        {
            "model": result["model"],
            "closed_set_test_accuracy": result["closed_set"]["test"]["accuracy"],
            "closed_set_test_macro_f1": result["closed_set"]["test"]["macro_f1"],
            "verification_test_eer": result["verification"]["test"]["eer"],
            "verification_test_min_dcf": result["verification"]["test"]["min_dcf"],
            "open_set_test_known_id_accuracy": result["open_set"]["test"][
                "known_identification_accuracy"
            ],
            "open_set_test_unknown_rejection": result["open_set"]["test"][
                "unknown_rejection_rate"
            ],
            "open_set_test_auroc": result["open_set"]["test"][
                "known_unknown_auroc"
            ],
        }
        for result in results
    ]


In [ ]:
"""Validation-ranked, weight-only rolling ECAPA checkpoints."""

import json
from pathlib import Path
from typing import Any, Mapping

import torch


TASK_PATHS = {
    "closed": "best_closed.pt",
    "verification": "best_verification.pt",
    "open": "best_open.pt",
    "balanced": "best_balanced.pt",
    "latest": "latest.pt",
}


def balanced_score(metrics: Mapping[str, float]) -> float:
    return (float(metrics["closed_macro_f1"]) + (1.0 - float(metrics["verification_eer"])) + float(metrics["open_auroc"])) / 3.0


def rank_metrics(task: str, metrics: Mapping[str, float], epoch: int, *, frozen_closed_macro_f1: float | None = None, max_closed_f1_drop: float = .005):
    """Return comparable rank; lower wins. Earlier epoch wins exact ties."""
    if task == "closed":
        return (-float(metrics["macro_f1"]), -float(metrics["accuracy"]), int(epoch))
    if task == "verification":
        return (float(metrics["eer"]), float(metrics["min_dcf"]), int(epoch))
    if task == "open":
        return (-float(metrics["known_unknown_auroc"]), -float(metrics["dir_at_far_1pct"]), int(epoch))
    if task == "balanced":
        if frozen_closed_macro_f1 is not None and float(metrics["closed_macro_f1"]) < frozen_closed_macro_f1 - max_closed_f1_drop:
            return None
        return (-balanced_score(metrics), int(epoch))
    raise ValueError(f"Unknown checkpoint task: {task}")


def checkpoint_payload(encoder: Mapping[str, Any], epoch: int, task: str, validation_metrics: Mapping[str, Any], training_config: Mapping[str, Any], classifier: Mapping[str, Any] | None = None) -> dict[str, Any]:
    return {
        "encoder": dict(encoder),
        "classifier": dict(classifier or {}),
        "epoch": int(epoch),
        "task": str(task),
        "validation_metrics": dict(validation_metrics),
        "training_config": dict(training_config),
    }


class CheckpointManager:
    def __init__(self, directory: str | Path, max_checkpoints: int = 5):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.max_checkpoints = int(max_checkpoints)
        self.registry_path = self.directory / "checkpoint_registry.json"
        self.registry = json.loads(self.registry_path.read_text()) if self.registry_path.exists() else {}

    def save_if_improved(self, task: str, payload: Mapping[str, Any], metrics: Mapping[str, float], epoch: int, *, reason: str = "validation improvement", frozen_closed_macro_f1: float | None = None, max_closed_f1_drop: float = .005) -> bool:
        rank = rank_metrics(task, metrics, epoch, frozen_closed_macro_f1=frozen_closed_macro_f1, max_closed_f1_drop=max_closed_f1_drop)
        if rank is None or (task in self.registry and tuple(self.registry[task]["rank"]) <= tuple(rank)):
            return False
        path = self.directory / TASK_PATHS[task]
        torch.save(dict(payload), path)
        self.registry[task] = {"task": task, "epoch": int(epoch), "validation_metrics": dict(metrics), "path": str(path), "selection_reason": reason, "rank": list(rank)}
        self._write_registry()
        self.enforce_limit()
        return True

    def record_latest(self, payload: Mapping[str, Any], epoch: int) -> None:
        torch.save(dict(payload), self.directory / TASK_PATHS["latest"])
        self.registry["latest"] = {"task": "latest", "epoch": int(epoch), "path": str(self.directory / TASK_PATHS["latest"]), "selection_reason": "latest epoch"}
        self._write_registry()
        self.enforce_limit()

    def enforce_limit(self) -> None:
        allowed = {TASK_PATHS[key] for key in TASK_PATHS}
        for path in self.directory.glob("*.pt"):
            if path.name not in allowed:
                path.unlink()
        paths = sorted(self.directory.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
        for path in paths[self.max_checkpoints:]:
            path.unlink()

    def _write_registry(self) -> None:
        self.registry_path.write_text(json.dumps(self.registry, indent=2, default=str) + "\n", encoding="utf-8")


In [ ]:
# Protocol helper must exist before frozen validation and training cells.
def protocol_vectors(cache, protocol, path_column="audio_path"):
    paths = protocol[path_column].astype(str)
    missing = sorted(set(paths) - set(cache))
    assert not missing, f"Missing cached protocol embeddings: {missing[:3]}"
    return np.stack([cache[str(path)] for path in paths])


## 7. Frozen ECAPA baseline


In [ ]:
print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM)

pretrained = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/kaggle/working/pretrained_ecapa",
    run_opts={"device": str(DEVICE)},
    freeze_params=True,
)
pretrained.eval()


In [ ]:
def evaluation_loader(frame):
    return DataLoader(
        VoxVietnamAudioDataset(frame, training=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        collate_fn=collate_audio,
    )


@torch.inference_mode()
def extract_pretrained_embeddings(
    frame, description, encoder_classifier=pretrained
):
    features = []
    for waveforms, lengths, _, _ in tqdm(
        evaluation_loader(frame), desc=description, leave=False
    ):
        embeddings = encoder_classifier.encode_batch(
            waveforms.to(DEVICE, non_blocking=True),
            wav_lens=lengths.to(DEVICE, non_blocking=True),
            normalize=False,
        ).squeeze(1)
        embeddings = F.normalize(embeddings.float(), p=2, dim=-1)
        features.append(embeddings.cpu().numpy())
    result = np.concatenate(features)
    assert result.shape == (len(frame), 192)
    assert np.isfinite(result).all()
    return result


print("Frozen ECAPA initialized")


## 8. Execute frozen ECAPA evaluation


In [ ]:
frozen_embedding_matrix = extract_pretrained_embeddings(
    evaluation_df, "Frozen ECAPA: cache every unique audio", pretrained
)
assert evaluation_df["audio_path"].is_unique
frozen_embedding_cache = dict(zip(
    evaluation_df["audio_path"].astype(str), frozen_embedding_matrix
))
assert len(frozen_embedding_cache) == len(evaluation_df)

frozen_closed_validation = evaluate_closed_validation(
    protocol_vectors(frozen_embedding_cache, closed_train_protocol), closed_train_protocol["speaker_id"],
    protocol_vectors(frozen_embedding_cache, closed_valid_protocol), closed_valid_protocol["speaker_id"],
)
frozen_validation_metrics = frozen_closed_validation["metrics"]
frozen_summary = pd.DataFrame([{
    "model": "frozen ECAPA-TDNN",
    "validation_closed_macro_f1": frozen_validation_metrics["macro_f1"],
    "validation_closed_accuracy": frozen_validation_metrics["accuracy"],
}])
display(frozen_summary)


## 9. Fine-tuning model, classifier, and DataLoader


In [ ]:
train_speakers = sorted(train_df[LABEL_COL].astype(str).unique())
speaker_to_id = {
    speaker: index for index, speaker in enumerate(train_speakers)
}
NUM_SPEAKERS = len(speaker_to_id)


def collate_train(batch):
    waveforms, valid_samples, _, speakers = zip(*batch)
    waveforms = torch.stack(waveforms)
    lengths = torch.tensor(valid_samples, dtype=torch.float32)
    lengths = lengths / waveforms.shape[1]
    labels = torch.tensor(
        [speaker_to_id[speaker] for speaker in speakers], dtype=torch.long
    )
    return waveforms, lengths, labels


train_dataset = VoxVietnamAudioDataset(train_df, training=True)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    collate_fn=collate_train,
)

print("Train audio:", len(train_dataset))
print("Train speakers:", NUM_SPEAKERS)
print("Train batches:", len(train_loader))


In [ ]:
class ECAPAFineTuner(nn.Module):
    def __init__(self, pretrained_model, num_speakers):
        super().__init__()
        self.compute_features = pretrained_model.mods.compute_features
        self.feature_normalizer = pretrained_model.mods.mean_var_norm
        self.encoder = pretrained_model.mods.embedding_model
        self.classifier = Classifier(
            input_size=192, out_neurons=num_speakers
        )

    def forward(self, waveforms, lengths):
        features = self.compute_features(waveforms)
        features = self.feature_normalizer(features, lengths)
        embeddings = self.encoder(features, lengths)
        scores = self.classifier(embeddings)
        return scores, embeddings


model = ECAPAFineTuner(pretrained, NUM_SPEAKERS).to(DEVICE)
pretrained.mods.classifier.to("cpu")
if "mean_var_norm_emb" in pretrained.mods:
    pretrained.mods.mean_var_norm_emb.to("cpu")

criterion = LogSoftmaxWrapper(
    AdditiveAngularMargin(margin=0.2, scale=30)
).to(DEVICE)

for parameter in model.encoder.parameters():
    parameter.requires_grad_(False)

optimizer = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters(), "lr": ENCODER_LEARNING_RATE},
        {"params": model.classifier.parameters(), "lr": CLASSIFIER_LEARNING_RATE},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


## 10. Training and validation functions


In [ ]:
def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def train_one_epoch(epoch, warmup_epochs):
    model.train()
    if epoch < warmup_epochs:
        model.encoder.eval()
    optimizer.zero_grad(set_to_none=True)
    losses, predictions, targets = [], [], []

    progress = tqdm(train_loader, desc=f"Train {epoch + 1}", leave=False)
    for step, (waveforms, lengths, labels) in enumerate(progress):
        waveforms = waveforms.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast_context():
            scores, _ = model(waveforms, lengths)
            loss = criterion(scores, labels.unsqueeze(1))

        scaler.scale(loss / GRAD_ACCUM).backward()
        update = (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(train_loader)
        if update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        predictions.extend(scores.squeeze(1).argmax(-1).detach().cpu().tolist())
        targets.extend(labels.detach().cpu().tolist())
        losses.append(float(loss.detach().cpu()))
        progress.set_postfix(loss=f"{np.mean(losses):.4f}")

    return {
        "train_loss": float(np.mean(losses)),
        "train_macro_f1": float(f1_score(
            targets, predictions, average="macro", zero_division=0
        )),
    }


@torch.inference_mode()
def extract_finetuned_embeddings(frame, description):
    model.eval()
    features = []
    for waveforms, lengths, _, _ in tqdm(
        evaluation_loader(frame), desc=description, leave=False
    ):
        waveforms = waveforms.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        with autocast_context():
            _, embeddings = model(waveforms, lengths)
        embeddings = F.normalize(
            embeddings.squeeze(1).float(), p=2, dim=-1
        )
        features.append(embeddings.cpu().numpy())
    result = np.concatenate(features)
    assert result.shape == (len(frame), 192)
    assert np.isfinite(result).all()
    return result


def validate_verification(epoch):
    embeddings = extract_finetuned_embeddings(
        valid_df, f"Validation {epoch + 1}"
    )
    scores = score_trials(valid_df, embeddings, valid_trials)
    return verification_curve(valid_trials["label"], scores)


## 11. Pre-training memory probes


In [ ]:
process = psutil.Process(os.getpid())


def memory_snapshot(label):
    message = f"{label}: RAM={process.memory_info().rss / 1024**3:.2f} GiB"
    if USE_AMP:
        message += (
            f", GPU allocated={torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
            f", reserved={torch.cuda.memory_reserved() / 1024**3:.2f} GiB"
        )
    print(message)


memory_snapshot("Before probes")
for debug_step, debug_batch in enumerate(train_loader):
    if debug_step >= 20:
        break
print("DataLoader probe passed")

model.train()
model.encoder.eval()
optimizer.zero_grad(set_to_none=True)
for debug_step, (waveforms, lengths, labels) in enumerate(train_loader):
    waveforms = waveforms.to(DEVICE)
    lengths = lengths.to(DEVICE)
    labels = labels.to(DEVICE)
    with autocast_context():
        scores, embeddings = model(waveforms, lengths)
        debug_loss = criterion(scores, labels.unsqueeze(1))
    scaler.scale(debug_loss).backward()
    optimizer.zero_grad(set_to_none=True)
    assert scores.shape == (len(labels), 1, NUM_SPEAKERS)
    assert embeddings.shape == (len(labels), 1, 192)
    assert torch.isfinite(debug_loss)
    if debug_step >= 5:
        break

del waveforms, lengths, labels, scores, embeddings, debug_loss, debug_batch
gc.collect()
if USE_AMP:
    torch.cuda.empty_cache()
memory_snapshot("After probes")
print("Forward/backward probe passed")


## 12. Execute fine-tuning; select checkpoint on validation EER/minDCF


In [ ]:
training_config = {
    "dataset": EXPECTED_DATASET,
    "dataset_root": str(DATASET_ROOT),
    "device": str(DEVICE),
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "segment_seconds": SEGMENT_SECONDS,
    "train_audio": len(train_df),
    "train_speakers": NUM_SPEAKERS,
    "validation_audio": len(valid_df),
    "validation_trials": len(valid_trials),
    "test_audio": len(test_df),
    "test_trials": len(test_trials),
    "selection_metric": "validation-only closed F1, verification EER, open AUROC, balanced score",
    "max_epochs": MAX_EPOCHS,
    "warmup_epochs": WARMUP_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "encoder_learning_rate": ENCODER_LEARNING_RATE,
    "classifier_learning_rate": CLASSIFIER_LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "checkpoint_eval_interval": CHECKPOINT_EVAL_INTERVAL,
    "max_rolling_checkpoints": MAX_ROLLING_CHECKPOINTS,
    "max_closed_f1_drop": MAX_CLOSED_F1_DROP,
}
TRAINING_CONFIG_PATH.write_text(
    json.dumps(training_config, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(training_config, indent=2))


In [ ]:
checkpoint_manager = CheckpointManager(CHECKPOINT_DIR, MAX_ROLLING_CHECKPOINTS)
bad_epochs = 0
history = []

for epoch in range(MAX_EPOCHS):
    if epoch == WARMUP_EPOCHS:
        for parameter in model.encoder.parameters():
            parameter.requires_grad_(True)
        print("ECAPA encoder unfrozen")

    row = {"epoch": epoch + 1}
    row.update(train_one_epoch(epoch, WARMUP_EPOCHS))
    validation = validate_verification(epoch)
    row.update({f"valid_{key}": value for key, value in validation.items()})
    validation_cache = extract_finetuned_embeddings(
        evaluation_df, f"Validation checkpoint epoch {epoch + 1}"
    )
    validation_vectors = dict(zip(evaluation_df["audio_path"].astype(str), validation_cache))
    closed_validation = evaluate_closed_validation(
        protocol_vectors(validation_vectors, closed_train_protocol), closed_train_protocol["speaker_id"],
        protocol_vectors(validation_vectors, closed_valid_protocol), closed_valid_protocol["speaker_id"],
    )["metrics"]
    verification_validation = evaluate_verification_validation(
        validation_vectors, valid_enrollment_protocol, valid_trials
    )["metrics"]
    open_validation = evaluate_open_validation(
        validation_vectors, open_valid_gallery, open_valid_queries
    )["metrics"]
    row.update({f"valid_closed_{key}": value for key, value in closed_validation.items()})
    row.update({f"valid_open_{key}": value for key, value in open_validation.items()})
    history.append(row)
    print(row)

    scheduler.step(validation["eer"])
    compact_payload = checkpoint_payload(
        model.encoder.state_dict(), epoch + 1, "latest",
        {"verification": verification_validation, "closed": closed_validation, "open": open_validation},
        training_config, model.classifier.state_dict(),
    )
    checkpoint_manager.record_latest(compact_payload, epoch + 1)
    checkpoint_manager.save_if_improved("closed", checkpoint_payload(model.encoder.state_dict(), epoch + 1, "closed", closed_validation, training_config, model.classifier.state_dict()), closed_validation, epoch + 1)
    checkpoint_manager.save_if_improved("verification", checkpoint_payload(model.encoder.state_dict(), epoch + 1, "verification", verification_validation, training_config, model.classifier.state_dict()), verification_validation, epoch + 1)
    checkpoint_manager.save_if_improved("open", checkpoint_payload(model.encoder.state_dict(), epoch + 1, "open", open_validation, training_config, model.classifier.state_dict()), open_validation, epoch + 1)
    balanced_metrics = {"closed_macro_f1": closed_validation["macro_f1"], "verification_eer": verification_validation["eer"], "open_auroc": open_validation["known_unknown_auroc"]}
    balanced_rank = rank_metrics("balanced", balanced_metrics, epoch + 1, frozen_closed_macro_f1=frozen_validation_metrics["macro_f1"], max_closed_f1_drop=MAX_CLOSED_F1_DROP)
    if balanced_rank is not None:
        checkpoint_manager.save_if_improved("balanced", checkpoint_payload(model.encoder.state_dict(), epoch + 1, "balanced", balanced_metrics, training_config, model.classifier.state_dict()), balanced_metrics, epoch + 1, frozen_closed_macro_f1=frozen_validation_metrics["macro_f1"], max_closed_f1_drop=MAX_CLOSED_F1_DROP)
    if balanced_rank is not None:
        bad_epochs = 0
    else:
        bad_epochs += 1

    pd.DataFrame(history).to_csv(TRAINING_HISTORY_PATH, index=False)
    pd.DataFrame(history).to_csv(CHECKPOINT_DIR / "checkpoint_validation_history.csv", index=False)
    memory_snapshot(f"End epoch {epoch + 1}")
    if bad_epochs >= EARLY_STOPPING_PATIENCE:
        print("Early stopping")
        break


## 13. Restore checkpoint and cache fine-tuned embeddings


In [ ]:
selected_balanced_path = BEST_BALANCED_PATH
if not selected_balanced_path.is_file():
    print("No balanced epoch satisfied closed-F1 constraint; falling back to best closed checkpoint")
    selected_balanced_path = BEST_CLOSED_PATH
CHECKPOINT = selected_balanced_path
assert CHECKPOINT.is_file(), f"Checkpoint not found: {CHECKPOINT}"
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
model.encoder.load_state_dict(checkpoint["encoder"])
model.classifier.load_state_dict(checkpoint["classifier"])
model.to(DEVICE).eval()

finetuned_embedding_matrix = extract_finetuned_embeddings(
    evaluation_df, "Fine-tuned ECAPA: cache every unique audio"
)
assert evaluation_df["audio_path"].is_unique
finetuned_embedding_cache = dict(zip(
    evaluation_df["audio_path"].astype(str), finetuned_embedding_matrix
))
assert len(finetuned_embedding_cache) == len(evaluation_df)
print(f"Cached {len(evaluation_df)} fine-tuned audio embeddings")


## 14. Execute fine-tuned ECAPA evaluation and comparison


In [ ]:
frozen_results = evaluate_three_tasks(
    "frozen ECAPA-TDNN",
    frozen_embedding_cache,
    THREE_TASK_PROTOCOLS,
    output_dir=FROZEN_EVALUATION_DIR,
)
finetuned_results = evaluate_three_tasks(
    "fine-tuned ECAPA-TDNN",
    finetuned_embedding_cache,
    THREE_TASK_PROTOCOLS,
    output_dir=FINETUNED_EVALUATION_DIR,
)
finetuned_summary = pd.DataFrame(
    summarize_three_task_results([finetuned_results])
)
display(finetuned_summary)

three_task_summary = pd.DataFrame(
    summarize_three_task_results([frozen_results, finetuned_results])
)
three_task_summary.to_csv(SUMMARY_PATH, index=False)
display(three_task_summary)


## Output artifacts

- `ecapa_voxvietnam_best.pt`
- `ecapa_training_config.json`
- `ecapa_training_history.csv`
- `evaluation/frozen/{verification,closed_set,open_set}_metrics.json`
- `evaluation/finetuned/{verification,closed_set,open_set}_metrics.json`
- Model-specific trial scores, predictions, and confusion matrix under
  each evaluation directory.
- `three_task_summary.csv`

These paths are an artifact contract, not claimed results. Populate them
only by running this notebook on gated VoxVietnam audio with a Kaggle GPU.
Never tune a checkpoint, SVM hyperparameter, or threshold on test scores.
